# Day 010 · 正态分布与金融
**Normal Distribution** · 阶段 P1 · 量化基础

> 正态分布是金融教科书最爱的工具。但 1987 年标普一天跌 22.6%(黑色星期一),按正态分布算十的几十次方年才一次,而它两百年里已发生四五次。沪深三百 2024-10-9 和 2025-4-7 两次跌 7%(均为 -6.37σ 事件),按正态算一万年一次,我们五年里发生了两次。这一节用沪深三百、标普五百、纳斯达克、比特币四个标的真实数据,讲透正态在金融中段为什么有效、在尾部为什么完全失效;讲清 68-95-99.7 法则、偏度、峰度、Q-Q 图四个工具;讲清 LTCM 1998 / 雷曼 2008 两个被正态滥用搞死的真实案例。学完你会对任何'假设收益服从正态'的金融模型,本能在心里画一个问号——它在中段可能漂亮,但兜不住尾巴。

---

**课件生成日期:** 2026-05-04  ·  **建议学习时长:** 20 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


⏳ 缺少 1 个包,正在自动安装:['yfinance']
✓ 安装完成
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 理解正态分布(钟形曲线)的两个参数:均值 + 标准差,以及它在自然界为何无处不在
- 记牢 68-95-99.7 法则,以及它在金融中段实测大致成立、在尾部大幅失效
- 用偏度(对称性)+ 峰度(厚尾度)量化分布形状,知道正态偏度=0 / 超额峰度=0
- 学会画 Q-Q 图(分位数对比图)一眼识别厚尾,十秒判断能不能套用经典模型
- 理解 LTCM 1998 和雷曼 2008 是怎么被正态分布的滥用搞死的
- 掌握散户应对厚尾的三条铁律:不满仓 / 留备用金 / 大跌别摊平

## 历史背景:Black Monday、LTCM 和金融业最伟大也最危险的工具

1809 年 Carl Friedrich Gauss 在《天体运动理论》里第一次系统化正态分布,用来描述天文测量的误差,因此正态分布也叫高斯分布。19 世纪 Quetelet 把它搬到社会统计(身高、智商),20 世纪 Bachelier、Black-Scholes-Merton 把它搬到金融——他们假设股票收益服从正态分布,这成了整个现代金融工程的地基。

**1987 年 10 月 19 日 Black Monday**:标普五百一天跌 22.6%。如果之前 5 年日波动率约 1%,这是一个 22σ 事件,按正态分布算概率约 10⁻¹⁰⁹——比宇宙原子总数倒数还小。正态分布告诉你这事一辈子都不该看到一次,但金融市场后来又出现了 1998、2008、2020、2022 多次类似冲击。

**1998 年 LTCM 危机**:对冲基金 Long-Term Capital Management,合伙人里有 Myron Scholes 和 Robert Merton 两位诺奖得主。他们的策略全部基于'收益服从正态'的假设。1998 年 8 月俄罗斯主权债务违约,他们模型里'永远不可能'的极端事件连发,几个月内净值蒸发 90%,需要美联储召集 14 家投行救市。

**2008 年金融危机**:雷曼倒闭那段时间,所有华尔街投行的风险模型几乎都基于正态。结果半年内出现了几十次模型说'一辈子不该出现'的极端日。雷曼自己就是这么倒的。

**2024-10-9 和 2025-4-7**:沪深三百两次跌 7%,均为 -6.37σ 事件,按正态算一万年一次,半年内发生两次。前者是国庆后政策预期落空,后者是关税战预期升级。叙事级事件,正态根本预测不了。

本节最重要的两句话:**正态分布最大的危险不是它错,是它太好用,好用到让所有人都忘了它的边界**;**它解释了 99% 的中段,但能让你账户归零的就是它解释不了的那 1%**。

**关键人物:**
- Carl Friedrich Gauss(1809,正态分布数学奠基)
- Adolphe Quetelet(19世纪,把正态搬到社会统计)
- Louis Bachelier(1900,首次用正态建模股价)
- Black、Scholes、Merton(1973,期权定价公式建立在正态假设上)
- Benoit Mandelbrot(1963,首次质疑金融正态假设,提出幂律分布)
- Myron Scholes & Robert Merton(LTCM 1998,亲身验证正态假设的灾难)

## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 钟形曲线 + 两个参数

正态分布是大自然里最常见的'中间多两头少'的形状。1000 个成年男性身高,绝大部分集中在 170cm 附近,150cm 和 200cm 都极少——画柱状图就是钟形。

正态分布完全被两个参数确定:
**均值 μ**:钟形最高点的位置(平均水平)
**标准差 σ**:钟形的胖瘦(数据散得多开)

金融数据:沪深三百过去 5 年日收益均值 ≈ 0,标准差 ≈ 1.1%;标普 ≈ 1%;纳斯达克 ≈ 1.4%;比特币 ≈ 3%(是沪深三百的 3 倍)。这些都是'看似正态'的初步描述,但中段像不代表尾部像。

```
f(x) = (1/σ√(2π)) × exp(-(x-μ)²/(2σ²))
```

> **举例:** 正态分布只需要 μ 和 σ 两个数就完全确定。这个简洁性是它统治金融工程 100 年的核心原因。学生 t 分布(明天讲)多了一个自由度参数,数学只复杂一点,但拟合精度提升一个数量级。


### 2. 68-95-99.7 法则:中段成立,尾部失效

如果一组数据是正态分布,那么:

- **68.3%**:的数据落在 ±1σ 内
- **95.4%**:的数据落在 ±2σ 内
- **99.7%**:的数据落在 ±3σ 内



- **真实数据验证(沪深三百近 5 年 1200+ 个交易日)**

实测 ±1σ:76.9%(理论 68.3%)— 中段比理论还集中,大致吻合
实测 ±2σ:95.5%(理论 95.4%)— 完美
实测 ±3σ:98.8%(理论 99.7%)— 看似只差 0.9%,但意味着 ±3σ 外的天数,实测 ~15 天 vs 理论 ~3.4 天,**实测是理论的 4 倍多**


- **±5σ 极端日**:正态预测 0.0007 天(基本一辈子碰不到),实测沪深三百 4 天 / 标普 3 天 / 纳斯达克 2 天 / 比特币 1 天。**实测是理论的几千倍**。这就是金融数据的厚尾。

```
P(|X-μ| ≤ kσ) ≈ {0.683, 0.954, 0.997} for k = 1, 2, 3
```

> **举例:** 沪深三百实测 ±1σ 76.9%,比理论值 68.3% 还高(中段更集中),但 ±3σ 实测 98.8% < 理论 99.7%(尾部更厚)。这是金融数据的典型形态:中段更集中 + 尾部更厚,看起来像正态但骨子里完全不是。


### 3. 为什么金融人爱正态:数学方便 + 中心极限定理

**两大好处:**

**① 数学上极简**。正态分布完全被 μ 和 σ 确定,几乎所有经典金融定理(Markowitz MPT、CAPM、Black-Scholes 期权定价)都建立在收益正态假设上,推导极其简洁。

**② 中心极限定理(CLT)给的心理安慰**:很多独立的小波动加起来,无论原始分布多奇怪,聚合后会越来越像正态。这给金融人一个安慰——好像不管原始数据多奇怪,聚合一下就能用正态近似。

**但金融市场恰恰是最不喜欢被正态约束的地方。** 三个原因:
① 金融数据不独立(波动率聚集 — Day 7 讲过)
② 金融数据不同分布(平静期和危机期是两种生物)
③ 极端事件由叙事驱动,不是随机扰动(政策、关税、暴雷不是高斯噪声)

中段 CLT 还能用,尾部完全不能。这是今天要打破的最大幻觉。

```
CLT: 当 N→∞,(X̄_N - μ)·√N / σ → N(0, 1)
```

> **举例:** 沪深三百日数据峰度 7.15(尾巴远比正态厚),但按 21 天聚合后峰度降到 4.74,63 天聚合后峰度降到 0.63(几乎正态)。CLT 真的在工作,但散户的交易是日级的——月度像正态对你也没意义,你今天止损被打掉是 1 天的事。


### 4. 偏度 + 峰度:测量分布的'变形度'

**偏度(Skewness)** 衡量对称性:
正态偏度 = 0(完美对称)
负偏度 → 左尾巴长(大跌的概率比大涨高)— 大多数股票指数是这种
正偏度 → 右尾巴长(大涨的概率比大跌高)— 大宗商品偶尔出现

**峰度(Kurtosis,超额峰度)** 衡量尾部厚度:
正态超额峰度 = 0
正超额峰度 → 厚尾(中间更尖,尾巴更厚)
负超额峰度 → 薄尾

**实测值**:
沪深三百峰度 7.15,标普 7.1,纳斯达克 9+,比特币 3.73
(这些数字都远大于 0,说明四个标的全部是厚尾)

意义:两个收益序列可以标准差完全一样,但峰度差 10 倍——前者会让账户突然蒸发,后者大致按教科书走。光看标准差 / Sharpe 不看峰度,你就在裸奔。

```
偏度 = E[(X-μ)³] / σ³,    超额峰度 = E[(X-μ)⁴] / σ⁴ - 3
```

> **举例:** 比特币偏度有时 +1+(右偏,大涨概率高),有时 -1+(左偏,大跌概率高),随市场情绪剧烈摇摆。这种'偏度也不稳定'的特性是加密最危险的地方,也是为什么传统正态模型在加密上完全不能用。


### 5. Q-Q 图(分位数对比图):10 秒识别厚尾

Q-Q 图是量化界判断分布形态的'瑞士军刀'。

怎么画:横轴 = 正态分布的理论分位数,纵轴 = 实测数据的分位数,每个数据点对应一个 (理论 q, 实测 q)。

**怎么读**:
如果数据是正态,所有点会规规矩矩落在 45° 直线上
如果有厚尾,两端的点会偏离直线——**左下角点比直线低,右上角点比直线高**,形状像一条 S 曲线或者两端翘起的'弓'

**金融实测**:
沪深三百 Q-Q 图两端明显翘起,典型厚尾
标普五百 Q-Q 图两端翘起,但比沪深略缓
纳斯达克 Q-Q 图右上角异常高(科技股大涨多)
**比特币 Q-Q 图两端直接甩出图外**——根本不是同一个游戏

实战:你拿到任何一个新策略的收益序列,**第一件事就画 Q-Q 图**,十秒钟就能判断能不能套用经典正态模型。这是量化研究员的本能动作。

```
理论 q_i = Φ⁻¹((i-0.5)/N),  实测 q_i = sorted_data[i]
```

> **举例:** scipy.stats.probplot(returns, dist='norm', plot=plt) 一行画出 Q-Q 图。看到两端不在直线上立刻警觉——你的模型假设可能不成立,需要换学生 t 或更复杂的工具。


## 实操:正态分布与金融 · 直方图 + 拟合 + Q-Q 图 + 极端日清单

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_010_normal_distribution.py — 沪深三百/标普/纳斯达克/比特币 厚尾真相
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats

tickers = {
    '沪深三百':  '000300.SS',
    '标普五百':  '^GSPC',
    '纳斯达克':  '^IXIC',
    '比特币':    'BTC-USD',
}

# 每个标的单独下载,避免不同交易日历强行 dropna 把样本清空
returns = {}
for name, sym in tickers.items():
    raw = yf.download(sym, period='5y', auto_adjust=True, progress=False)
    if raw is None or raw.empty:
        print(f'× {name} 拿不到数据'); continue
    s = raw['Close']
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    r = s.pct_change().dropna() * 100  # 百分比收益
    if len(r) < 100:
        print(f'× {name} 样本太少 {len(r)}'); continue
    returns[name] = r
    print(f'✓ {name}: {len(r)} 天 ({r.index.min().date()} ~ {r.index.max().date()})')

names = list(returns.keys())
print()

# === 1) 各标的描述统计:均值/标准差/偏度/峰度 ===
print('=== 五年日收益描述统计(单位:%)===')
print(f'{"标的":<10}{"均值":>8}{"标准差":>10}{"偏度":>8}{"峰度":>10}')
for name in names:
    r = returns[name]
    print(f'{name:<10}{r.mean():>8.3f}{r.std():>10.2f}{stats.skew(r):>8.2f}{stats.kurtosis(r):>10.2f}')
print()
print('注:正态分布的偏度=0,峰度=0(超额峰度)。峰度越大说明尾巴越厚。')

# === 2) 六十八九十五九十九点七法则 实测 vs 理论 ===
print('\n=== 六十八九十五九十九点七法则:实测 vs 理论 ===')
print(f'{"标的":<10}{"|z|<1":>10}{"|z|<2":>10}{"|z|<3":>10}')
print(f'{"理论值":<10}{"68.3%":>10}{"95.4%":>10}{"99.7%":>10}')
for name in names:
    r = returns[name]
    z = (r - r.mean()) / r.std()
    p1 = (np.abs(z) < 1).mean() * 100
    p2 = (np.abs(z) < 2).mean() * 100
    p3 = (np.abs(z) < 3).mean() * 100
    print(f'{name:<10}{p1:>9.1f}%{p2:>9.1f}%{p3:>9.1f}%')

# === 3) 极端日:超过 3σ / 5σ 的天数 ===
print('\n=== 超过 3σ 与 5σ 的天数 ===')
print(f'{"标的":<10}{">3σ 实测":>12}{"正态预测":>12}{">5σ 实测":>12}{"正态预测":>14}')
for name in names:
    r = returns[name]
    z = (r - r.mean()) / r.std()
    n_total = len(z)
    n3 = int((np.abs(z) > 3).sum())
    n5 = int((np.abs(z) > 5).sum())
    e3 = n_total * 2 * (1 - stats.norm.cdf(3))
    e5 = n_total * 2 * (1 - stats.norm.cdf(5))
    print(f'{name:<10}{n3:>10}天{e3:>10.1f}天{n5:>10}天{e5:>12.4f}天')

# === 4) 各标的史上五大单日跌幅 ===
print('\n=== 五年内单日最大五次跌幅 ===')
for name in names:
    r = returns[name]
    worst = r.nsmallest(5)
    print(f'\n{name}:')
    for d_, v in worst.items():
        zv = (v - r.mean()) / r.std()
        print(f'  {d_.date()}: {v:+.2f}% ({zv:+.2f}σ)')

# === 5) 直方图 + 正态拟合 + Q-Q 图,2 行 N 列布局 ===
n = len(names)
fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
if n == 1:
    axes = axes.reshape(2, 1)
for i, name in enumerate(names):
    r = returns[name]
    mu, sigma = float(r.mean()), float(r.std())
    if not np.isfinite(mu) or not np.isfinite(sigma) or sigma <= 0:
        continue
    ax = axes[0, i]
    ax.hist(r, bins=80, density=True, alpha=0.55, color='#3aa0ff', label='实际')
    x = np.linspace(float(r.min()), float(r.max()), 500)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=1.6, label='正态拟合')
    ax.axvline(mu - 3*sigma, ls='--', color='gray', lw=0.8)
    ax.axvline(mu + 3*sigma, ls='--', color='gray', lw=0.8)
    ax.set_title(f'{name} 日收益分布', fontsize=11)
    ax.set_xlabel('日收益 (%)'); ax.set_ylabel('密度')
    ax.set_xlim(mu - 6*sigma, mu + 6*sigma)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax2 = axes[1, i]
    stats.probplot(r, dist='norm', plot=ax2)
    ax2.set_title(f'{name} Q-Q 图', fontsize=11)
    ax2.get_lines()[0].set_markersize(3)
    ax2.get_lines()[0].set_alpha(0.5)
    ax2.get_lines()[1].set_color('red')
    ax2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('day010_normal_qq.png', dpi=120)
plt.close(fig)

# === 6) 中心极限定理直觉:不重叠分块求和 1/5/21/63 日 ===
print('\n=== 中心极限定理:聚合周期越长,越像正态(不重叠分块)===')
ref_name = '沪深三百' if '沪深三百' in returns else names[0]
rr = returns[ref_name].values  # numpy array,百分比单位
for window in [1, 5, 21, 63]:
    n_blocks = len(rr) // window
    blocks = rr[:n_blocks * window].reshape(n_blocks, window).sum(axis=1)
    z = (blocks - blocks.mean()) / blocks.std()
    print(f'  {window:>2} 日聚合: 样本 {n_blocks:>4d}  偏度 {stats.skew(z):+.2f}  峰度 {stats.kurtosis(z):+.2f}')

fig2, ax = plt.subplots(figsize=(9, 5))
for window, color in zip([1, 5, 21, 63], ['#ff5c5c', '#ffaa33', '#3aa0ff', '#22c55e']):
    n_blocks = len(rr) // window
    blocks = rr[:n_blocks * window].reshape(n_blocks, window).sum(axis=1)
    z = (blocks - blocks.mean()) / blocks.std()
    ax.hist(z, bins=40, density=True, alpha=0.45, label=f'{window} 日聚合', color=color)
x = np.linspace(-5, 5, 500)
ax.plot(x, stats.norm.pdf(x, 0, 1), 'k--', lw=1.5, label='标准正态')
ax.set_xlim(-5, 5); ax.set_xlabel('标准化后收益 (z)'); ax.set_ylabel('密度')
ax.set_title(f'{ref_name} — 聚合周期越长 越像正态(中心极限定理直觉)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('day010_clt.png', dpi=120)
plt.close(fig2)

✓ 沪深三百: 1210 天 (2021-05-06 ~ 2026-04-30)
✓ 标普五百: 1255 天 (2021-05-05 ~ 2026-05-04)
✓ 纳斯达克: 1255 天 (2021-05-05 ~ 2026-05-04)
✓ 比特币: 1826 天 (2021-05-05 ~ 2026-05-04)

=== 五年日收益描述统计(单位:%)===
标的              均值       标准差      偏度        峰度
沪深三百         0.001      1.11    0.20      7.15
标普五百         0.049      1.07    0.16      7.11
纳斯达克         0.059      1.42    0.21      5.51
比特币          0.065      2.91   -0.03      3.75

注:正态分布的偏度=0,峰度=0(超额峰度)。峰度越大说明尾巴越厚。

=== 六十八九十五九十九点七法则:实测 vs 理论 ===
标的             |z|<1     |z|<2     |z|<3
理论值            68.3%     95.4%     99.7%
沪深三百           76.9%     95.5%     98.8%
标普五百           75.4%     95.1%     98.9%
纳斯达克           74.1%     95.3%     99.0%
比特币            77.6%     94.4%     98.0%

=== 超过 3σ 与 5σ 的天数 ===
标的              >3σ 实测        正态预测      >5σ 实测          正态预测
沪深三百              15天       3.3天         4天      0.0007天
标普五百              14天       3.4天         3天      0.0007天
纳斯达克              12天       3.4天         2天      0.0007天
比特币     

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 美股极端日(教科书案例) | S&P 500 1987-10-19 (Black Monday) | 一天跌 22.6%,按之前 σ ≈ 1% 算,这是 22σ 事件,正态预测概率 10⁻¹⁰⁹。比宇宙原子总数倒数还小,正态告诉你一辈子不该看到。但 200 年里发生了 4-5 次类似量级冲击。 |
| A 股极端日 | 沪深三百 2024-10-9 / 2025-4-7 | 两次单日跌 7%,均为 -6.37σ 事件。正态预测一万年一次,半年内发生两次。前者国庆后政策预期落空,后者关税战预期升级。叙事级事件正态根本预测不了。 |
| 加密极端日 | BTC 2022-6-13 / 2022-11-9 / 2026-2-5 | 三次单日跌幅 -16% / -14% / -14%,均超过 5σ。比特币加密市场 24h 不停 + 高杠杆 + 流动性薄,尾部远比股市厚,Mt.Gox 暴雷、Luna 归零、FTX 崩塌都是黑天鹅。 |
| LTCM 1998 | LTCM 长期资本管理公司 | 诺奖团队全部基于正态假设的策略,1998 年 8 月俄罗斯违约引爆连锁,几个月内净值蒸发 90%,需要美联储召集 14 家投行救市。正态滥用最经典的反面案例。 |
| 中心极限定理实测 | 沪深三百多窗口聚合 | 日数据峰度 7.15(厚尾)→ 21 天聚合峰度 4.74(中等厚尾)→ 63 天聚合峰度 0.63(几乎正态)。CLT 真的工作,但散户交易是日级,月度像正态对你也没意义。 |


## 常见坑

### ⚠ 01. 把过去最大回撤当未来上限

看到自己策略过去 3 年最大回撤 -10%,就以为未来撑死也是这个数。厚尾意味着未来一定有更深的——历史从来不是上限。正确做法:用学生 t 分布 / 极值理论(EVT)估计 99% 分位数风险。

### ⚠ 02. 以为高频数据更接近正态

正好相反。日内高频数据厚尾更严重,因为短时间的流动性踩踏比日级数据剧烈得多。这是为什么做高频策略的对冲基金(Renaissance、Two Sigma)风控比低频严格得多。

### ⚠ 03. 把样本均值当总体均值

你 3 年算出沪深三百日均收益 0.001%,这个数字本身有巨大误差。下一个 3 年可能完全不一样。做夏普比率 / 阿尔法估计时一定要看置信区间和 t-stat。

### ⚠ 04. 只看标准差不看偏度峰度

两个收益序列可以标准差完全相同,但峰度差 10 倍,风险天差地别。前者偶尔暴跌让账户清零,后者大致稳定。光看 Sharpe 不看峰度 = 裸奔。

### ⚠ 05. 在险价值(VaR)按正态算,严重低估真实风险

VaR 是华尔街最爱的风险指标。如果按正态算,沪深三百 99% VaR ≈ -2.5%/天,听起来安全。但实际过去 5 年最大单日跌幅 -7%,如果你按正态 VaR 配仓 + 加杠杆,大跌那天直接爆仓。实战要用历史模拟法 / Cornish-Fisher 修正 / 极值理论。

## 实战 SOP · 正态分布在金融的使用 SOP

1. 正态只在中段(±2σ 内)大致成立,尾部(±3σ 外)完全失效
2. 做风险估计永远不要只用正态,用学生 t / EVT / 历史模拟做交叉验证
3. 拿到新数据第一件事画 Q-Q 图,10 秒识别厚尾
4. VaR 用正态算会严重低估真实风险,实战要历史模拟 + Cornish-Fisher 修正
5. 看一只股票要算偏度 + 峰度,不能只看标准差
6. 中心极限定理在长周期(>60 天聚合)可用,但散户交易是日级,日级 ≠ 正态
7. 永远假设单日跌幅可能是正态预测的 3-5 倍,所以仓位永远不要满到极限

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 正态分布:钟形曲线,均值 μ + 标准差 σ 两个参数完全确定,中段成立,尾部失效。
3. 68-95-99.7 法则:中段实测 ≈ 理论,但 ±3σ 外实测是理论的 4 倍,±5σ 外是几千倍。
4. 金融四标的实测峰度:沪深三百 7.15 / 标普 7.1 / 纳斯达克 9+ / 比特币 3.73。全部厚尾。
5. 金融人爱正态:数学方便(只需 μ+σ)+ CLT 心理安慰。但日级金融数据不满足 CLT 前提。
6. Q-Q 图是量化界十秒识别厚尾的'瑞士军刀',两端翘起 = 厚尾。
7. VaR 用正态算严重低估真实风险。LTCM 1998 / 雷曼 2008 / 沪深 -7% 都是血泪教训。
8. 散户三条铁律:① 永远不要满仓(留 20% 浮动);② 永远准备 20% 黑天鹅基金;③ 大跌别摊平。
9. 正态分布最大危险不是它错,是它太好用——好用到让所有人都忘了它的边界。

## 自测题

**Q1.** 解释:为什么沪深三百实测 ±1σ 比例 76.9% 比理论 68.3% 还高,但 ±3σ 比例 98.8% 比理论 99.7% 低?

**Q2.** 你看到一只新基金过去 3 年 Sharpe 2.5,标准差 8%。你买之前还要看哪两个数字?为什么?

**Q3.** Q-Q 图上,如果一只标的的点呈现'两端翘起的 S 形',这说明什么?该用什么分布替代正态?

**Q4.** 如果你按正态分布算的 99% VaR 给账户配杠杆,真实极端日发生时为什么会爆仓?算给我看。

**Q5.** 中心极限定理告诉我们长周期会收敛到正态。这对'做月度策略 + 算 Sharpe'的散户有什么实战意义?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 011 · 厚尾与黑天鹅** (Fat Tails & Black Swans)

Day 11:厚尾与黑天鹅 — 既然正态不行,拿什么替代?我们引入学生 t 分布(只多一个自由度参数,精度提升一个数量级)、幂律分布、塔勒布的反脆弱思想、凯利公式、杠铃策略。学完你就有了应对极端行情的工具箱。

## 推荐阅读

- Mandelbrot《The Variation of Certain Speculative Prices》(1963)— 首次质疑金融正态假设,幂律分布
- Taleb《The Black Swan》(2007)— 正态分布滥用的批判经典,黑天鹅思想入门
- Lowenstein《When Genius Failed: The Rise and Fall of LTCM》(2000)— 正态被滥用搞死诺奖团队的全程
- Cont《Empirical Properties of Asset Returns: Stylized Facts and Statistical Issues》(2001)— 金融数据 12 个'风格化事实',包括厚尾
- scipy.stats 文档 — Q-Q 图、KS 检验、Anderson-Darling 检验三件套,Python 量化标配